## STEP 1 Knowledge Graph 기초

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
client = OpenAI()
df = pd.read_excel("Articles_20260305_125404.xlsx")

text = df.iloc[0]['content']

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 형태의 트리플을 추출하세요.  각 줄에 하나씩, 형식: (주어, 관계, 목적어)
  최대 10개만 추출하세요."""},
        {"role": "user", "content": text}
    ]
)

print("=== 추출된 트리플 ===")
print(response.choices[0].message.content)

=== 추출된 트리플 ===
1. (당내 강경파, 개최, 대법원장 탄핵 공청회)
2. (정청래, 압박, 조 대법원장 사퇴)
3. (민주당, 악화, 갈등)
4. (조희대 대법원장, 표명, 반대의사)
5. (정청래, 촉구, 조 대법원장 자진사퇴)
6. (조 대법원장, 심사숙고, 사법개혁 3법 처리)
7. (강경파 의원들, 열다, 조 대법원장 탄핵을 위한 공청회)
8. (여권 의원들, 촉구, 조 대법원장 탄핵)
9. (민 의원, 주장, 대법원장이 책임지고 물러나야 한다)
10. (당 지도부, 거리 두다, 조 대법원장 탄핵)


In [2]:
all_triples = []

for idx in range(5):  # 기사 5개만
    text = df.iloc[idx]['content']
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 트리플을 추출하세요.
  각 줄에: (주어, 관계, 목적어)
  최대 5개만."""},
            {"role": "user", "content": text}
        ]
    )

    triples_text = response.choices[0].message.content
    print(f"\n--- 기사 {idx}: {df.iloc[idx]['title'][:30]} ---")
    print(triples_text)

    # 파싱
    for line in triples_text.strip().split("\n"):
        line = line.strip().strip("()")
        parts = [p.strip().strip("()") for p in line.split(",")]
        if len(parts) == 3:
            all_triples.append(tuple(parts))

print(f"\n총 트리플 수: {len(all_triples)}")


--- 기사 0: 조희대 탄핵 꺼낸 與… ‘사법 3법’ 갈등 확산 ---
1. (더불어민주당, 사퇴 압박, 조희대 대법원장)
2. (정청래, 자진사퇴 촉구, 조희대 대법원장)
3. (당내 강경파, 탄핵 거론, 조희대 대법원장)
4. (전현희, 탄핵 촉구, 조희대 대법원장)
5. (당 지도부, 탄핵 논의하지 않음, 조희대 대법원장)

--- 기사 1: 김 총리 "2차 공공기관 이전, 수도권 잔류 최소화·나 ---
1. (김민석 국무총리, 방침을 명확히 했다, '나눠먹기식' 분산 배치)
2. (김 총리, 열고, 제10회 국가정책조정회의)
3. (2차 공공기관 이전, 목적, 지역 성장 엔진 다극화)
4. (정부, 방침, 수도권 잔류 최소화)
5. (김 총리, 집적화하겠다, 지역이 실질적 성장 거점이 되도록)

--- 기사 2: 檢 직격한 이 대통령...與 "가짜 진술로 만든 공소, ---
1. (한병도, 발언하다, "정의를 실현해야 할 검찰이 오히려 국가 권력으로 사람을 죽이려 한 행위는 절대 용납할 수 없다")
2. (한병도, 말하다, "정치 검찰의 사건 조작은 강도 살인보다 더 나쁜 국가적 범죄")
3. (한병도, 밝히다, "쌍방울 대북 송금 수사의 실태는 경악을 넘어 분노를 자아낸다")
4. (이 대통령, 언급하다, "검찰에 의한 증거 조작 가능성")
5. (한 병도, 다짐하다, "국정조사를 통해 이 조작의 설계자들을 반드시 심판대 앞에 세우도록 하겠다")

--- 기사 3: 김정은, 이란戰 반응… 최현號 ‘2격능력’ 과시 ---
1. (김정은 북한 국무위원장, 실시한, 해상대지상 전략순항미사일 시험발사)
2. (조선중앙통신, 보도했다, 김 위원장이 함의 성능 및 작전 수행능력 평가 시험공정을 파악했다)
3. (김 위원장, 수행되고 있다, 해군의 핵무장화)
4. (시험발사, 공개됐다, 최소 4발의 전략순항미사일)
5. (북한, 과시한, 순항미사일 연속 발사 능력)

--- 기사 4: “尹어게인 말려달라” 野 의원 편지에… 尹 “충정 알겠 ---
1. (윤석열

In [3]:
from collections import Counter

# 모든 주어와 목적어 수집
entities = []
for s, r, o in all_triples:
    entities.append(s)
    entities.append(o)

counter = Counter(entities)
print("=== 자주 등장하는 엔티티 ===")
for entity, count in counter.most_common(10):
    print(f"  {entity}: {count}회")

=== 자주 등장하는 엔티티 ===
  조희대 대법원장: 5회
  1. (더불어민주당: 1회
  2. (정청래: 1회
  3. (당내 강경파: 1회
  4. (전현희: 1회
  5. (당 지도부: 1회
  1. (김민석 국무총리: 1회
  '나눠먹기식' 분산 배치: 1회
  2. (김 총리: 1회
  제10회 국가정책조정회의: 1회
